## Import packages

In [ ]:
import sys
sys.path.append('..')

import json
import numpy as np
import matplotlib.pyplot as plt
from src import create_data_generators, get_class_weights
from src import build_baseline_cnn
from src import train_model
from src import evaluate_model, plot_confusion_matrix, plot_roc_curve, plot_training_history

plt.style.use('seaborn-v0_8-whitegrid')
plt.rcParams.update({
    'figure.facecolor': 'white',
    'axes.facecolor': 'white',
    'axes.labelcolor': 'black',
    'xtick.color': 'black',
    'ytick.color': 'black',
    'text.color': 'black',
    'legend.facecolor': 'white',
    'legend.edgecolor': 'black'
})

## Load data with augmentation

In [ ]:
data_dir = '../data/chest_xray'
train_gen, val_gen, test_gen = create_data_generators(
    data_dir, target_size=(224, 224), batch_size=32, augment=True
)

## Get class weights

In [ ]:
class_weights = get_class_weights(train_gen)

## Build baseline CNN

In [ ]:
model = build_baseline_cnn(input_shape=(224, 224, 3))
model.summary()

## Train baseline model

In [ ]:
import os
os.makedirs('../models', exist_ok=True)

history = train_model(
    model,
    train_gen, val_gen,
    epochs=30,
    class_weights=class_weights,
    model_save_path='../models/baseline_model.keras'
)

## Plot training history

In [ ]:
plot_training_history(history, title="Baseline CNN")

## Delete remove

In [ ]:
from tensorflow.keras.models import load_model
model = load_model('../models/baseline_model.keras')
print("Loaded saved baseline model")

## Evaluate on test set

In [ ]:
results = evaluate_model(model, test_gen)

## Confusion matrix

In [ ]:
plot_confusion_matrix(
    results['y_true'], results['y_pred'],
    results['class_names'],
    title="Baseline CNN - Confusion Matrix"
)

## ROC curve

In [ ]:
auc_score = plot_roc_curve(
    results['y_true'], results['y_prob'],
    title="Baseline CNN - ROC Curve"
)

## Save baseline results

In [ ]:
baseline_results = {
    "model": "baseline_cnn",
    "accuracy": results['accuracy'],
    "precision": results['precision'],
    "recall": results['recall'],
    "f1": results['f1'],
    "auc": auc_score
}

with open('../models/baseline_results.json', 'w') as f:
    json.dump(baseline_results, f, indent=2)

print("Baseline results saved to ../models/baseline_results.json")

## Summary

In [ ]:
print("=" * 50)
print("BASELINE CNN SUMMARY")
print("=" * 50)
print(f"Accuracy:  {results['accuracy']:.4f} ({results['accuracy'] * 100:.2f}%)")
print(f"Precision: {results['precision']:.4f}")
print(f"Recall:    {results['recall']:.4f}")
print(f"F1-Score:  {results['f1']:.4f}")
print(f"AUC:       {auc_score:.4f}")
print("=" * 50)
print("\nNext: 04_1_transfer_learning_resnet50.ipynb")